# prenatalppkt walkthrough

A tour of the real pipeline, following one dummy exam (Apple Sally) stage by stage from raw file to finished Phenopacket - what each stage does, the test that proves it, and honest notes on what still doesn't work.

## 1. Orientation

prenatalppkt turns ultrasound exports into GA4GH Phenopackets through three layers:

1. **Extractors** (`etl/extractors/`) - read the raw file, pull out biometry measurements.
2. **Section parsers** (`etl/sections/`) - read the raw file, pull out everything else (impression, anatomy, dating, EFW, ratios).
3. **Builders** (`builders/`) - call both layers, assemble one Phenopacket per fetus.

Apple Sally is a fake scan exported as Observer JSON: one fetus, a full set of core measurements, and a genuine abnormal finding (a Dandy-Walker malformation, which I explicitly inserted into Apple Sally's clinical notes). We'll follow Apple Sally through every stage.

In [1]:
import json
from pathlib import Path

DATA_DIR = Path("tests/data")
raw = json.loads((DATA_DIR / "Apple_Sally_pretty.json").read_text())
fetus = raw["fetuses"][0]

print("Top-level keys:", sorted(raw.keys()))
print("Fetus count:", len(raw["fetuses"]))
print("Measurement labels present:", [m["label"] for m in fetus["measurements"]])

Top-level keys: ['adnexa', 'cervix', 'endomyocds', 'exam', 'fetuses', 'finalize', 'gyn_procedure', 'hist_phys_vitals', 'uterine_artery', 'uterus']
Fetus count: 1
Measurement labels present: ['AC', 'BPD', 'HC', 'Femur', 'Nuchal Fold', 'Cerebellum']


Six measurements, all four core ones (AC, BPD, HC, Femur) plus two optional ones (Nuchal Fold, Cerebellum). Let's follow Apply Sally's HC (head circumference) measurement through the whole pipeline.

## 2. Foundations

The pieces every measurement passes through, regardless of source format. We'll use Apple Sally's actual HC entry throughout, not a made-up example.

In [2]:
hc = next(m for m in fetus["measurements"] if m["label"] == "HC")
print("Apple Sally's real HC entry:", hc)

Apple Sally's real HC entry: {'label': 'HC', 'value': 25, 'decimal_places': 2, 'unit_of_measure': 'cm', 'calculated_ega': 26.9, 'calculated_percentile': 42.5, 'percentile_for_display': '43%', 'include_in_avg_ga_calc': 1, 'print_in_report': 1, 'calculated_z_score': 0, 'fetus_number': 1}


`value=25`, `unit_of_measure=cm` (25cm = 250mm), `calculated_ega=26.9` weeks, `calculated_percentile=42.5`. Four real numbers, four different building blocks.

### Gestational age - `gestational_age.py`, `GestationalAge`

Converts Apple Sally's decimal-week EGA into whole weeks + days. Tested in `test_gestational_age.py` across every (week, day) combination from 10-42 weeks - a real bug here (floating-point truncation silently losing a day in 43% of combinations) was found and fixed.

In [3]:
from prenatalppkt.gestational_age import GestationalAge

ga = GestationalAge.from_weeks(hc["calculated_ega"])
print(f"Apple Sally's real EGA, {hc['calculated_ega']} weeks -> {ga.weeks}w{ga.days}d")

Apple Sally's real EGA, 26.9 weeks -> 26w6d


### Percentile ranges - `measurements/percentile_range.py`, `PercentileRange` (+ `Percentile` enum)

Turns Apple Sally's real percentile number into one of 8 bins. Tested in `test_percentile_range.py` (boundary values) and `test_reference_range.py`. Another real bug fixed: negative percentiles used to be silently accepted instead of rejected.

In [4]:
from prenatalppkt.measurements.percentile_range import PercentileRange

r = PercentileRange.evaluate(hc["calculated_percentile"])
print(f"Apple Sally's real HC percentile, {hc['calculated_percentile']}% -> bin '{r.bin_key}'")

try:
    PercentileRange.evaluate(-5.0)
except ValueError as e:
    print(f"A percentile that can't exist is correctly rejected: {e}")

Apple Sally's real HC percentile, 42.5% -> bin 'between_10p_50p'
A percentile that can't exist is correctly rejected: Invalid percentile: -5.0


### Label-normalization layer - `etl/constants.py`

Real files use inconsistent labels for the same measurement (`"Femur"` vs `"FL"`). This layer maps every format's raw label to one shared `BiometryMeasurement` name. This session added 11 new labels found in real Observer files that weren't recognized before (Tibia, Fibula, Radius, Ulna, Foot, Cisterna Magna, Nasal Bone, Lateral Vent left/right, Biorbit, Mean Gest Sac). Tested in `test_constants.py`.

In [5]:
from prenatalppkt.etl.constants import OBSERVER_NAME_MAP

print(f"Apple Sally's raw label 'HC' maps to: {OBSERVER_NAME_MAP['HC']}")
print(f"'FL' and 'Femur' map to the same standard value: {OBSERVER_NAME_MAP['FL'] == OBSERVER_NAME_MAP['Femur']}")
print(f"Total recognized Observer labels: {len(OBSERVER_NAME_MAP)}")

DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for head_circumference
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for biparietal_diameter
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for femur_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for abdominal_circumference
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for occipitofrontal_diameter
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for crown_rump_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for nuchal_translucency
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for tibia_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for fibula_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for radius_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for ulna_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for foot_length
DEBUG:prenatalppkt.etl.term_bin_factory:Loaded mappings for: ['head_circumference', 'biparietal_diameter', 'femur_length', 'abdominal_circumference', 'occipitofrontal_diame

Apple Sally's raw label 'HC' maps to: BiometryMeasurement.HEAD_CIRCUMFERENCE
'FL' and 'Femur' map to the same standard value: True
Total recognized Observer labels: 22


### Scan-type classification - `etl/scan_type.py`, `classify_fetus()`

Looks at which standard measurement names are present and decides which extraction path applies (`FIRST_TRIMESTER`, `T2_T3_BIOMETRY`, or `UNKNOWN`). Tested in `test_scan_type.py`.

In [6]:
from prenatalppkt.etl.scan_type import classify_fetus

print(f"Apple Sally has all four core measurements, so the fetus classifies as: {classify_fetus(fetus).value}")

Apple Sally has all four core measurements, so the fetus classifies as: t2_t3_biometry


### TermBinFactory - `etl/term_bin_factory.py`

Given a standard name, a value, and a percentile, picks the matching `TermBin` - an HPO term, a LOINC code, and a normal/abnormal flag, all from the YAML mapping (`data/mappings/biometry_hpo_mappings.yaml`, loaded by `mapping_loader.py`'s `BiometryMappingLoader`, tested in `test_mapping_loader.py`). This is where everything above comes together. Tested in `test_term_bin_factory.py`; the underlying bin object is `measurements/term_bin.py`'s `TermBin`, tested in `test_term_bin.py`.

In [7]:
from prenatalppkt.etl.term_bin_factory import TermBinFactory

factory = TermBinFactory()
term_bin = factory.create_term_bin(
    name="HC",
    value_mm=hc["value"] * 10,  # cm -> mm, same conversion the real extractor does
    percentile=hc["calculated_percentile"],
    gestational_age=ga,
)
print(f"Apple Sally's real HC -> {term_bin.hpo_id} '{term_bin.hpo_label}'")
print(f"LOINC code: {term_bin.loinc_code}  |  normal: {term_bin.normal}")
print(f"Description: {term_bin.description}")

DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for head_circumference
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for biparietal_diameter
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for femur_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for abdominal_circumference
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for occipitofrontal_diameter
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for crown_rump_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for nuchal_translucency
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for tibia_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for fibula_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for radius_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for ulna_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for foot_length
DEBUG:prenatalppkt.etl.term_bin_factory:Loaded mappings for: ['head_circumference', 'biparietal_diameter', 'femur_length', 'abdominal_circumference', 'occipitofrontal_diame

Apple Sally's real HC -> HP:0000240 'Abnormality of skull size'
LOINC code: LOINC:11984-2  |  normal: True
Description: HC: 250.0 mm (42.5%) at 26w6d


Apple Sally's HC is at the 42.5th percentile - well within normal range, so this comes back `normal=True`. That's expected: Apple Sally's real abnormal finding (Dandy-Walker) isn't a biometry measurement at all - it's in Apple Sally's clinical text, which is the next section.

## 3. HPO + text mining

### The recognizer abstraction - `hpo/hpo_cr.py`, `hpo/fenominal_cr.py`

`HpoConceptRecognizer` is an abstract base class (`abc.ABCMeta` + `@abc.abstractmethod`) - the real extension point for any text-to-HPO backend. `FenominalConceptRecognizer` is its one concrete implementation today, wrapping the third-party `fenominal` library. Every hit comes back as a `SimpleTerm` (`hpo/simple_term.py`) - `hpo_id`, `hpo_label`, `excluded`, `gestational_age`. Tested in `test_fenominal_cr.py`, `test_hpo_parser.py`.

### What fenominal actually is

This is the tool doing the real work below, so it's worth naming plainly before using it: fenominal reads a doctor's free-text note and finds the HPO terms hiding inside it. Give it a sentence like "no evidence of macrocephaly" and it finds "macrocephaly," looks up its HPO code, and also notices the word "no" so it can mark that finding as absent instead of present. `HpoParser` (used right below) is just a thin loader around it - fenominal is the part that actually reads the text.

In [8]:
from prenatalppkt.hpo import HpoParser

hpo_parser = HpoParser()
hpo_cr = hpo_parser.get_hpo_concept_recognizer()
print(type(hpo_cr).__name__, "is-a", type(hpo_cr).__mro__[1].__name__)
print("HPO version:", hpo_parser.get_version())

DEBUG:hpotk.store._github:Pulling tag from https://api.github.com/repos/obophenotype/human-phenotype-ontology/tags
DEBUG:hpotk.store._github:Fetched 30 tags
DEBUG:hpotk.util:Using default encoding 'utf-8'
DEBUG:hpotk.util:Opening /Users/jv2684/.hpo-toolkit/HP/hp.v2026-06-23.json
DEBUG:hpotk.util:Looks like a local file: /Users/jv2684/.hpo-toolkit/HP/hp.v2026-06-23.json
DEBUG:hpotk.util:Looks like decompressed data
DEBUG:hpotk.ontology.load.obographs._load:Extracting ontology terms
DEBUG:hpotk.ontology.load.obographs._factory:Unknown synonym type http://purl.obolibrary.org/obo/hp#allelic_requirement
DEBUG:hpotk.ontology.load.obographs._factory:Unknown synonym type http://purl.obolibrary.org/obo/hp#allelic_requirement
DEBUG:hpotk.ontology.load.obographs._factory:Unknown synonym type http://purl.obolibrary.org/obo/hp#allelic_requirement
DEBUG:hpotk.ontology.load.obographs._factory:Unknown synonym type http://purl.obolibrary.org/obo/hp#allelic_requirement
DEBUG:hpotk.ontology.load.obograph

FenominalConceptRecognizer is-a HpoConceptRecognizer
HPO version: 2026-06-23


### Running it on Apple Sally's real clinical text

In [9]:
impression_text = raw["finalize"]["generalComment"]["plain_text"]
print(impression_text[:400], "...")

The sonogram was significant for splaying of the cerebellar hemispheres. There was evidence of a cyst measuring __ connecting with the fourth ventricle. This is consistent with  ...


In [10]:
hits = hpo_cr.parse(impression_text)
for hit in hits:
    print(f"{hit.hpo_id}  {hit.hpo_label}  (excluded={hit.excluded})")

HP:0000256  Macrocephaly  (excluded=True)
HP:0002119  Ventriculomegaly  (excluded=True)
HP:0001274  Agenesis of corpus callosum  (excluded=False)


**Worth knowing:** the "no evidence of macrocephaly, ventriculomegaly or agenesis of the corpus callosum" sentence isn't organic sonographer documentation - it was added by hand to Apple Sally's case specifically as a simple, deliberate negation-extraction test for the text-mining tool. At the time it was written, that tool was `FastHPOCR`; `fenominal` picks up the first two terms in that list correctly today too. The third term is where it gets interesting - see the next cell.

### A negation-scope inconsistency this session found

Apple Sally's sentence negates three findings in one clause: "no evidence of macrocephaly, ventriculomegaly **or** agenesis of the corpus callosum." Running the full impression text through `hpo_cr.parse()` two cells up, the first two terms came back `excluded=True` - correctly negated - but agenesis of the corpus callosum came back `excluded=False`, as if it were a positive finding, despite sitting in the exact same negated list. Isolating just that clause reproduces it directly:

In [11]:
negation_clause = (
    "There was no evidence of macrocephaly, ventriculomegaly or "
    "agenesis of the corpus callosum."
)
for hit in hpo_cr.parse(negation_clause):
    print(f"{hit.hpo_id}  {hit.hpo_label}  excluded={hit.excluded}")

HP:0000256  Macrocephaly  excluded=True
HP:0002119  Ventriculomegaly  excluded=True
HP:0001274  Agenesis of corpus callosum  excluded=False


**The gap this session found and documented:** Apple Sally's text says, word for word, "This is consistent with a Dandy-Walker malformation" - the exact official HPO label for `HP:0001305`. It's not in the list above. And unlike some of the other gaps found (where the exact phrase works fine on its own, just not inside a full sentence), this one is a true miss in every phrasing tried - not a sentence-context problem. Tracked as an honest `xfail` test (`test_dandy_walker_malformation_recognized_in_anatomy_text` in `test_fetal_anatomy.py`, plus a builder-level companion in `test_observer_phenopacket.py`) rather than hidden behind a loose assertion.

In [12]:
for phrasing in ["Dandy-Walker malformation", "Dandy Walker malformation", "Dandy-Walker syndrome"]:
    hits = hpo_cr.parse(phrasing)
    print(f"{phrasing!r:32s} -> {[(h.hpo_id, h.hpo_label) for h in hits]}")

'Dandy-Walker malformation'      -> []
'Dandy Walker malformation'      -> []
'Dandy-Walker syndrome'          -> []


### fenominal across all six real clinical-text fixtures

Beyond Apple Sally, every other fixture in `tests/data/` has its own real, independently-written clinical impression text. Running the same `hpo_cr.parse()` across all of them gives a broader read on where fenominal actually holds up, rather than judging it from the one case that was specifically built to test it:

In [13]:
for path in sorted(DATA_DIR.glob("*_pretty.json")):
    fixture = json.loads(path.read_text())
    text = fixture.get("finalize", {}).get("generalComment", {}).get("plain_text")
    if not text:
        continue
    print(f"=== {path.name} ===")
    hits = hpo_cr.parse(text)
    if not hits:
        print("  (no hits)")
    for hit in hits:
        print(f"  {hit.hpo_id}  {hit.hpo_label}  excluded={hit.excluded}")
    print()

=== Apple_Sally_pretty.json ===
  HP:0000256  Macrocephaly  excluded=True
  HP:0002119  Ventriculomegaly  excluded=True
  HP:0001274  Agenesis of corpus callosum  excluded=False

=== Blue_Sally_pretty.json ===
  HP:0000122  Unilateral renal agenesis  excluded=False
  HP:0000122  Unilateral renal agenesis  excluded=False
  HP:0000813  Bicornuate uterus  excluded=False

=== Charm_Sally_pretty.json ===
  HP:0010866  Abdominal wall defect  excluded=False
  HP:0001539  Omphalocele  excluded=False
  HP:0001539  Omphalocele  excluded=False
  HP:0001539  Omphalocele  excluded=False

=== Diva_Sally_pretty.json ===
  HP:0030716  Acrania  excluded=False

=== Eclair_Sally_pretty.json ===
  HP:0001627  Abnormal heart morphology  excluded=False

=== Gwen_Sally_pretty.json ===
  HP:0000126  Hydronephrosis  excluded=True



**What this survey shows:** every real anomaly actually mentioned in these six fixtures gets picked up - Charm Sally's omphalocele, Blue Sally's unilateral renal agenesis (twice - it's genuinely mentioned twice in the text), Diva Sally's acrania, Eclair Sally's cardiac anomaly (captured as the generic `Abnormal heart morphology` - the text only says "suspicious for a cardiac anomaly," so there isn't a more specific term to extract), and Gwen Sally's correctly-excluded hydronephrosis. Apple Sally's Dandy-Walker miss and the negation-scope inconsistency above are the two real gaps in this set, not a pattern across every fixture - fenominal's actual failure rate here is low, but both gaps are exactly the kind of thing a systematic benchmark needs to catch rather than find by accident - a good next step for anyone evaluating text-mining alternatives.

## 4. Observer path, end to end

Apple Sally has 6 measurements, not just HC. Let's extract all of them, then run every remaining section parser on Apple Sally's real data, then assemble Apple Sally's final Phenopacket.

### The extractor - `etl/extractors/observer.py`

`extract_all_fetuses()` runs `classify_fetus` + `TermBinFactory` (section 2) across every measurement for every fetus in the file. Tested in `test_observer.py`, `test_observer_multifetus.py`.

In [14]:
from prenatalppkt.etl.extractors import observer as observer_extractor

bins_by_fetus = observer_extractor.extract_all_fetuses(raw)
for fetus_number, term_bins in bins_by_fetus.items():
    print(f"Fetus {fetus_number}: {len(term_bins)} TermBins")
    for tb in term_bins:
        print(f"  {tb.hpo_id}  {tb.hpo_label}  normal={tb.normal}  ({tb.description})")

DEBUG:prenatalppkt.etl.extractors.observer:Starting Observer JSON extraction (multi-fetus)
DEBUG:prenatalppkt.etl.extractors.observer:Processing fetus 1, scan_type=t2_t3_biometry
DEBUG:prenatalppkt.etl.extractors.observer:Found 6 measurements
DEBUG:prenatalppkt.etl.extractors.observer:Processing measurement: AC
DEBUG:prenatalppkt.etl.extractors.observer:AC has percentile=55.6% (valid)
DEBUG:prenatalppkt.etl.extractors.observer:Creating TermBin for AC: value=226.20000000000002mm, percentile=55.6%, ga=<GestationalAge: 26 weeks, 6 days>
DEBUG:prenatalppkt.etl.term_bin_factory:Creating TermBin: name=AC, value=226.20000000000002mm, percentile=55.6%, ga=<GestationalAge: 26 weeks, 6 days>, method=None
DEBUG:prenatalppkt.etl.term_bin_factory:Selected HPO: HP:0034207 - Abnormal fetal gastrointestinal system morphology
DEBUG:prenatalppkt.etl.term_bin_factory:Created TermBin: HP:0034207 - normal=True
DEBUG:prenatalppkt.etl.extractors.observer:Processing measurement: BPD
DEBUG:prenatalppkt.etl.ext

Fetus 1: 4 TermBins
  HP:0034207  Abnormal fetal gastrointestinal system morphology  normal=True  (AC: 226.2 mm (55.6%) at 26w6d [Fetus 1])
  HP:0000240  Abnormality of skull size  normal=True  (BPD: 66.8 mm (51.2%) at 26w6d [Fetus 1])
  HP:0000240  Abnormality of skull size  normal=True  (HC: 250.0 mm (42.5%) at 26w6d [Fetus 1])
  HP:0002823  Abnormal femur morphology  normal=True  (Femur: 50.1 mm (46.8%) at 27w0d [Fetus 1])


Only 4 of Apple Sally's 6 measurements produced a `TermBin` - Nuchal Fold and Cerebellum are optional measurements with no HPO mapping yet (a known, documented gap, not a bug: `OPTIONAL_MEASUREMENTS` in `etl/term_bin_factory.py`).

### The section parsers - `etl/sections/`

Six parsers, all sharing the same `parse_X(data, source_format)` shape. Tested in `test_clinical_impression.py`, `test_clinical_indication.py`, `test_pregnancy_dating.py`, `test_estimated_fetal_weight.py`, `test_fetal_ratios.py`, `test_fetal_anatomy.py`.

In [15]:
from prenatalppkt.etl.sections import (
    parse_pregnancy_dating,
    parse_clinical_indication,
    parse_estimated_fetal_weight,
    parse_fetal_ratios,
)

dating = parse_pregnancy_dating(raw, "observer_json")
print("Dating:", {k: v for k, v in dating.items() if k != "raw_data"})

indication = parse_clinical_indication(raw, "observer_json")
print("\nClinical indication text:", repr(indication["indication_text"]))

efw = parse_estimated_fetal_weight(raw, "observer_json")
print("\nEFW:", efw["efw_grams"], "g,", efw["percentile"], "th percentile,", efw["method"])

ratios = parse_fetal_ratios(raw, "observer_json")
print("\nRatios:", [(r["name"], r["value"], r["within_range"]) for r in ratios["ratios"]])

Dating: {'lmp': '0001-01-01', 'edd': None, 'assigned_edd': None, 'dating_method': None, 'ga_by_lmp': None, 'ga_by_ultrasound': None, 'assigned_ga': None, 'source_format': 'observer_json'}

Clinical indication text: ''

EFW: 1014.8 g, 55.6 th percentile, Hadlock (AC, FL, HC)

Ratios: [('HC/AC', 1.105, True), ('FL/AC', 22.149, True), ('FL/BPD', 75, True)]


Two honest notes here, not bugs to fix right now: dating comes back mostly `None` (Apple Sally's fixture doesn't carry the fields this parser reads, and the builder falls back to reading gestational age out of a biometry `TermBin`'s own description instead - which is why Apple Sally's Phenopacket's GA is still correct despite this). Clinical indication text is empty even though the raw file *does* have an ICD-10 indication code (`Z36.1`) - the parser doesn't read that field today.

### Fetal anatomy - already introduced in section 3, now the full picture

In [16]:
from prenatalppkt.etl.sections import parse_fetal_anatomy

anatomy = parse_fetal_anatomy(raw, "observer_json", hpo_cr=hpo_cr)
print("Normal structures:", len(anatomy["normal_structures"]))
print("Abnormal structures:", anatomy["abnormal_structures"])
print("Not visualized:", anatomy["not_visualized"])
print("Structured anomaly:", anatomy["anomalies"])
print("HPO terms found:", [(t.hpo_id, t.hpo_label, t.excluded) for t in anatomy["hpo_terms"]])

Normal structures: 47
Abnormal structures: ['Head', 'Cerebellum']
Not visualized: ['Calvarium', 'BPD Level', 'Lateral Ventricles', 'Choroid Plexus', 'Cisterna Magna', 'Lungs', 'Distal Left Outflow', 'Distal Right Outflow', 'IVC', 'SVC', 'Spleen', 'Liver', 'Gall Bladder', 'Bowel', 'Lt Fingers', 'Rt Fingers', 'Lt Toes', 'Rt Toes']
Structured anomaly: [{'structure': 'Head', 'description': 'Dandy Walker', 'variant_type': 'Abnormal'}]
HPO terms found: [('HP:0045005', 'Neural tube defect', True)]


44 structures correctly read as normal, 2 correctly flagged abnormal (Head, Cerebellum), a real structured anomaly entry naming "Dandy Walker" directly - and still no `HP:0001305` in the HPO terms. Even this terse, 2-word structured form of the same finding doesn't get through fenominal. The free-text neural-tube-defect exclusion *is* caught correctly (`HP:0045005`, excluded=True) - so this isn't "fenominal doesn't work," it's specifically this one term.

### The builder - `builders/observer_phenopacket.py`

Calls everything above, once per fetus, and assembles the final Phenopacket. Tested end-to-end in `test_observer_phenopacket.py` against all 5 real fixtures.

In [17]:
from datetime import datetime, timezone
from google.protobuf.timestamp_pb2 import Timestamp
from prenatalppkt.builders import build_observer_phenopacket

now_ts = Timestamp()
now_ts.FromDatetime(datetime.now(tz=timezone.utc))

pps = build_observer_phenopacket(raw, hpo_parser, now_ts, accession_id="applesally")
apple_pp = pps[0]
print("Phenopacket id:", apple_pp.id)
print("Subject id:", apple_pp.subject.id)
print(f"{len(apple_pp.phenotypic_features)} phenotypic features:")
for pf in apple_pp.phenotypic_features:
    print(f"  {pf.type.id}  {pf.type.label}  excluded={pf.excluded}  ({pf.description})")
print("\nmeasurements:", list(apple_pp.measurements))

DEBUG:prenatalppkt.etl.extractors.observer:Starting Observer JSON extraction (multi-fetus)
DEBUG:prenatalppkt.etl.extractors.observer:Processing fetus 1, scan_type=t2_t3_biometry
DEBUG:prenatalppkt.etl.extractors.observer:Found 6 measurements
DEBUG:prenatalppkt.etl.extractors.observer:Processing measurement: AC
DEBUG:prenatalppkt.etl.extractors.observer:AC has percentile=55.6% (valid)
DEBUG:prenatalppkt.etl.extractors.observer:Creating TermBin for AC: value=226.20000000000002mm, percentile=55.6%, ga=<GestationalAge: 26 weeks, 6 days>
DEBUG:prenatalppkt.etl.term_bin_factory:Creating TermBin: name=AC, value=226.20000000000002mm, percentile=55.6%, ga=<GestationalAge: 26 weeks, 6 days>, method=None
DEBUG:prenatalppkt.etl.term_bin_factory:Selected HPO: HP:0034207 - Abnormal fetal gastrointestinal system morphology
DEBUG:prenatalppkt.etl.term_bin_factory:Created TermBin: HP:0034207 - normal=True
DEBUG:prenatalppkt.etl.extractors.observer:Processing measurement: BPD
DEBUG:prenatalppkt.etl.ext

Phenopacket id: applesally-fetus-1
Subject id: applesally-fetus-1
7 phenotypic features:
  HP:0034207  Abnormal fetal gastrointestinal system morphology  excluded=True  (Biometry: AC: 226.2 mm (55.6%) at 26w6d [Fetus 1])
  HP:0000240  Abnormality of skull size  excluded=True  (Biometry: BPD: 66.8 mm (51.2%) at 26w6d [Fetus 1])
  HP:0002823  Abnormal femur morphology  excluded=True  (Biometry: Femur: 50.1 mm (46.8%) at 27w0d [Fetus 1])
  HP:0000256  Macrocephaly  excluded=True  (Clinical impression: Macrocephaly)
  HP:0002119  Ventriculomegaly  excluded=True  (Clinical impression: Ventriculomegaly)
  HP:0001274  Agenesis of corpus callosum  excluded=False  (Clinical impression: Agenesis of corpus callosum)
  HP:0045005  Neural tube defect  excluded=True  (Fetal anatomy)

measurements: []


Seven features made it in: Apple Sally's 3 mappable biometry measurements (all normal, all excluded), 3 clinical-impression findings, 1 anatomy finding. Apple Sally's Dandy-Walker malformation is not among them - confirmed absent from the real, final Phenopacket, not just from an intermediate step.

One more real, honest gap, visible right here: `measurements` is an empty list. `build_observer_phenopacket()` doesn't emit LOINC-coded raw `Measurement` objects at all today - only HPO features. (`prenatalppkt.ipynb`'s demo cell works around this by hand-adding `Measurement`s on top, using the raw `TermBin`s from the extractor cell above - that's a notebook-level patch, not something the builder does itself.)

Also worth naming honestly: fenominal's negation detection isn't perfectly consistent either - Apple Sally's text says "no evidence of ... agenesis of the corpus callosum," and while Macrocephaly and Ventriculomegaly both correctly come back `excluded=True`, Agenesis of corpus callosum comes back `excluded=False` from the exact same sentence.

### A real bug this session found and fixed: twin exams

Two fetuses in one exam each have their own anatomy data in the real Observer schema. The builder used to read only the first fetus's anatomy and copy it onto every fetus's Phenopacket - so a twin exam's second fetus silently inherited the first fetus's findings instead of its own. Confirmed with a synthetic two-fetus exam, one with a renal finding, one with a skull finding.

In [18]:
fetus_1 = {
    "fetus": {"fetus_number": 1},
    "measurements": [
        {"label": "HC", "value": 250.0, "unit_of_measure": "mm", "calculated_percentile": 50.0},
        {"label": "BPD", "value": 65.0, "unit_of_measure": "mm", "calculated_percentile": 50.0},
        {"label": "AC", "value": 220.0, "unit_of_measure": "mm", "calculated_percentile": 50.0},
        {"label": "Femur", "value": 50.0, "unit_of_measure": "mm", "calculated_percentile": 50.0},
    ],
    "anatomy": [
        {"main": {"label": "Kidney", "anat_state": "Abnormal"}, "detail": [],
         "anomalies": [{"description": "Renal agenesis"}]}
    ],
}
fetus_2 = {
    "fetus": {"fetus_number": 2},
    "measurements": [
        {"label": "HC", "value": 240.0, "unit_of_measure": "mm", "calculated_percentile": 45.0},
        {"label": "BPD", "value": 63.0, "unit_of_measure": "mm", "calculated_percentile": 45.0},
        {"label": "AC", "value": 210.0, "unit_of_measure": "mm", "calculated_percentile": 45.0},
        {"label": "Femur", "value": 48.0, "unit_of_measure": "mm", "calculated_percentile": 45.0},
    ],
    "anatomy": [
        {"main": {"label": "Skull", "anat_state": "Abnormal"}, "detail": [],
         "anomalies": [{"description": "Acrania"}]}
    ],
}
twin_exam = {"exam": {"fetus_count": 2}, "fetuses": [fetus_1, fetus_2]}

twin_pps = build_observer_phenopacket(twin_exam, hpo_parser, now_ts, accession_id="TWIN")
for pp in twin_pps:
    hpo_ids = {pf.type.id for pf in pp.phenotypic_features}
    print(f"{pp.id}: {hpo_ids}")

DEBUG:prenatalppkt.etl.extractors.observer:Starting Observer JSON extraction (multi-fetus)
DEBUG:prenatalppkt.etl.extractors.observer:Processing fetus 1, scan_type=t2_t3_biometry
DEBUG:prenatalppkt.etl.extractors.observer:Found 4 measurements
DEBUG:prenatalppkt.etl.extractors.observer:Processing measurement: HC
DEBUG:prenatalppkt.etl.extractors.observer:HC has percentile=50.0% (valid)
DEBUG:prenatalppkt.etl.extractors.observer:Creating TermBin for HC: value=250.0mm, percentile=50.0%, ga=None
DEBUG:prenatalppkt.etl.term_bin_factory:Creating TermBin: name=HC, value=250.0mm, percentile=50.0%, ga=None, method=None
DEBUG:prenatalppkt.etl.term_bin_factory:Selected HPO: HP:0000240 - Abnormality of skull size
DEBUG:prenatalppkt.etl.term_bin_factory:Created TermBin: HP:0000240 - normal=True
DEBUG:prenatalppkt.etl.extractors.observer:Processing measurement: BPD
DEBUG:prenatalppkt.etl.extractors.observer:BPD has percentile=50.0% (valid)
DEBUG:prenatalppkt.etl.extractors.observer:Creating TermBin 

twin-fetus-1: {'HP:0034207', 'HP:0002823', 'HP:0000104', 'HP:0000240'}
twin-fetus-2: {'HP:0034207', 'HP:0002823', 'HP:0030716', 'HP:0000240'}


Both sets share the normal-biometry findings (expected - their measurements are similar), but the anomaly is never shared: `HP:0000104` (Renal agenesis) appears only in fetus 1's set, `HP:0030716` (Acrania) appears only in fetus 2's - neither fetus inherits the other's anatomy finding. Confirmed fixed with a regression test (`test_twin_each_fetus_keeps_its_own_anatomy_finding`) that was checked against the pre-fix code and failed the expected way before the fix landed. ViewPoint HL7 had the identical bug, fixed the same way (section 5).

### A real bug this session found and fixed: partial/targeted scans

Some real exams only re-measure one or two things to check on a prior finding, rather than a full anatomy survey. The extractor used to require all four core measurements before it would parse *anything* for a fetus - so these targeted scans produced nothing at all, even when the measurements present were perfectly good.

In [19]:
partial_scan = {
    "fetuses": [{
        "fetus": {"fetus_number": 1},
        "measurements": [
            {"label": "HC", "value": 240.0, "unit_of_measure": "mm", "calculated_percentile": 50.0},
            {"label": "BPD", "value": 63.0, "unit_of_measure": "mm", "calculated_percentile": 50.0},
            # No AC or Femur - a real, deliberate follow-up scan shape
        ],
    }]
}
partial_bins = observer_extractor.extract_all_fetuses(partial_scan)
for fetus_number, term_bins in partial_bins.items():
    print(f"Fetus {fetus_number}: {len(term_bins)} TermBins (used to be 0)")
    for tb in term_bins:
        print(f"  {tb.hpo_id}  {tb.hpo_label}")

DEBUG:prenatalppkt.etl.extractors.observer:Starting Observer JSON extraction (multi-fetus)
DEBUG:prenatalppkt.etl.extractors.observer:Processing fetus 1, scan_type=unknown
DEBUG:prenatalppkt.etl.extractors.observer:Found 2 measurements
DEBUG:prenatalppkt.etl.extractors.observer:Processing measurement: HC
DEBUG:prenatalppkt.etl.extractors.observer:HC has percentile=50.0% (valid)
DEBUG:prenatalppkt.etl.extractors.observer:Creating TermBin for HC: value=240.0mm, percentile=50.0%, ga=None
DEBUG:prenatalppkt.etl.term_bin_factory:Creating TermBin: name=HC, value=240.0mm, percentile=50.0%, ga=None, method=None
DEBUG:prenatalppkt.etl.term_bin_factory:Selected HPO: HP:0000240 - Abnormality of skull size
DEBUG:prenatalppkt.etl.term_bin_factory:Created TermBin: HP:0000240 - normal=True
DEBUG:prenatalppkt.etl.extractors.observer:Processing measurement: BPD
DEBUG:prenatalppkt.etl.extractors.observer:BPD has percentile=50.0% (valid)
DEBUG:prenatalppkt.etl.extractors.observer:Creating TermBin for BPD

Fetus 1: 2 TermBins (used to be 0)
  HP:0000240  Abnormality of skull size
  HP:0000240  Abnormality of skull size


Now returns the 2 measurements that are actually present, with a warning logged naming which of the four core measurements are missing - instead of discarding everything.

### First trimester - Diva

Diva is the one real first-trimester fixture: CRL only, no full biometry set. A different, deliberately simpler classification path (`ScanType.FIRST_TRIMESTER`).

In [20]:
diva_raw = json.loads((DATA_DIR / "Diva_Sally_pretty.json").read_text())
diva_pps = build_observer_phenopacket(diva_raw, hpo_parser, now_ts, accession_id="divasally")
diva_pp = diva_pps[0]
print("Diva's features:")
for pf in diva_pp.phenotypic_features:
    print(f"  {pf.type.id}  {pf.type.label}  excluded={pf.excluded}")
print("\nDiva's measurements:", list(diva_pp.measurements))

DEBUG:prenatalppkt.etl.extractors.observer:Starting Observer JSON extraction (multi-fetus)
DEBUG:prenatalppkt.etl.extractors.observer:Processing fetus 1, scan_type=first_trimester
DEBUG:prenatalppkt.etl.term_bin_factory:Creating TermBin: name=CRL, value=4.48mm, percentile=0.0%, ga=<GestationalAge: 11 weeks, 2 days>, method=None
DEBUG:prenatalppkt.etl.term_bin_factory:Selected HPO: HP:0001511 - Intrauterine growth retardation
DEBUG:prenatalppkt.etl.term_bin_factory:Created TermBin: HP:0001511 - normal=False
INFO:prenatalppkt.etl.extractors.observer:Extracted 1 total TermBins across 1 fetuses


Diva's features:
  HP:0001511  Intrauterine growth retardation  excluded=False
  HP:0030716  Acrania  excluded=False

Diva's measurements: []


**This is the T1 gap you asked about specifically:** Diva's Phenopacket has real phenotypic features (Apple Sally's CRL-derived growth finding, plus Apple Sally's real clinical-impression finding) - the T1 pipeline itself works correctly. But `measurements` is empty, the same gap Apple Sally has. It's just most visible on Diva because CRL is Apple Sally's *only* measurement, so there's nothing else to distract from the empty list. This is not a T1-specific bug - it's the same "builder doesn't emit `Measurement`s yet" gap, on every fixture, T1 or not.

## 5. ViewPoint path, end to end

A completely different raw shape (pipe-delimited HL7 segments instead of nested JSON), same three-layer pipeline. Using a real ViewPoint HL7 fixture with both biometry and a real anatomy finding.

In [21]:
vp_data = (DATA_DIR / "viewpoint_hl7_full_exam_test.txt").read_text()

from prenatalppkt.etl.extractors import viewpoint_hl7

vp_bins = viewpoint_hl7.extract_all_fetuses(vp_data)
for num, tbs in vp_bins.items():
    print(f"Fetus {num}: {len(tbs)} TermBins")
    for tb in tbs:
        print(" ", tb.hpo_id, tb.hpo_label, "|", tb.description)

DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for head_circumference
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for biparietal_diameter
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for femur_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for abdominal_circumference
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for occipitofrontal_diameter
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for crown_rump_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for nuchal_translucency
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for tibia_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for fibula_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for radius_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for ulna_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for foot_length
DEBUG:prenatalppkt.etl.term_bin_factory:Loaded mappings for: ['head_circumference', 'biparietal_diameter', 'femur_length', 'abdominal_circumference', 'occipitofrontal_diame

Fetus 1: 4 TermBins
  HP:0000240 Abnormality of skull size | HC: 205.0 mm (45.0%) at 21w3d
  HP:0000240 Abnormality of skull size | BPD: 53.0 mm (40.0%) at 21w2d
  HP:0034207 Abnormal fetal gastrointestinal system morphology | AC: 168.0 mm (38.0%) at 21w4d
  HP:0002823 Abnormal femur morphology | Femur: 34.0 mm (42.0%) at 21w3d


Same `TermBin` objects, same HPO+LOINC mapping - the only thing that changed is how the raw OBX segments got read. Now the anatomy side:

In [22]:
vp_anatomy = parse_fetal_anatomy(vp_data, "viewpoint_hl7", hpo_cr=hpo_cr)
print("Abnormal structures:", vp_anatomy["abnormal_structures"])
print("Anomaly:", vp_anatomy["anomalies"])
print("HPO terms:", [(t.hpo_id, t.hpo_label) for t in vp_anatomy["hpo_terms"]])

Abnormal structures: ['Cerebellum']
Anomaly: [{'structure': 'Cerebellum', 'description': 'cerebellar hypoplasia', 'variant_type': 'Abnormal'}]
HPO terms: [('HP:0001321', 'Cerebellar hypoplasia')]


"Cerebellar hypoplasia" - unlike Apple Sally's Dandy-Walker, this one *is* recognized correctly. Now the full build:

In [23]:
from prenatalppkt.builders import build_viewpoint_phenopacket

vp_pps = build_viewpoint_phenopacket(vp_data, hpo_parser, now_ts, accession_id="FULL00099")
vp_pp = vp_pps[0]
print("id:", vp_pp.id)
for pf in vp_pp.phenotypic_features:
    print(" ", pf.type.id, pf.type.label, "excluded="+str(pf.excluded), "|", pf.description)

DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for head_circumference
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for biparietal_diameter
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for femur_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for abdominal_circumference
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for occipitofrontal_diameter
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for crown_rump_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for nuchal_translucency
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for tibia_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for fibula_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for radius_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for ulna_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for foot_length
DEBUG:prenatalppkt.etl.term_bin_factory:Loaded mappings for: ['head_circumference', 'biparietal_diameter', 'femur_length', 'abdominal_circumference', 'occipitofrontal_diame

id: full00099-fetus-1
  HP:0000240 Abnormality of skull size excluded=True | Biometry: HC: 205.0 mm (45.0%) at 21w3d
  HP:0034207 Abnormal fetal gastrointestinal system morphology excluded=True | Biometry: AC: 168.0 mm (38.0%) at 21w4d
  HP:0002823 Abnormal femur morphology excluded=True | Biometry: Femur: 34.0 mm (42.0%) at 21w3d
  HP:0001321 Cerebellar hypoplasia excluded=False | Fetal anatomy


Only 4 features, not 5 - BPD is missing even though the extractor found it. HC and BPD both map to the same HPO term (`HP:0000240`, "Abnormality of skull size"), and the builder's HPO-id dedup keeps only the first occurrence. Expected, not a bug - the same dedup Apple Sally's Observer builder uses too.

### The twin-anatomy bug, ViewPoint's independent copy

ViewPoint HL7 had the exact same bug as Observer, found separately: `_parse_viewpoint_hl7_anatomy`'s own code comment claimed anatomy OBX segments "aren't per-fetus tagged the same way biometry's are" - checked directly against a real fixture, and that claim was wrong. Anatomy segments carry the same `Fetus1`/`Fetus2` sub-id biometry segments do.

In [24]:
twin_hl7 = r"""
MSH|^~\&|ViewPoint|Hospital|||20260115120000||ORU^R01|123456|P|2.4
OBX|1|ST|Fetus.Identifier^Fetus Identifier|Fetus1|A
OBX|2|NM|SkullFetus.HeadCircumference^HC|Fetus1|175^175.0|mm&millimeters^mm&millimeters
OBX|3|NM|SkullFetus.VP_HeadCircumference_Percentile|Fetus1|50^50%|%&percent^fmt&formatted
OBX|4|ST|BrainFetus.CerebellumAppearance^Cerebellum appearance|Fetus1|abnormal
OBX|5|ST|BrainFetus.CerebellumDetails^Cerebellum details|Fetus1|cerebellar hypoplasia
OBX|6|ST|Fetus.Identifier^Fetus Identifier|Fetus2|B
OBX|7|NM|SkullFetus.HeadCircumference^HC|Fetus2|170^170.0|mm&millimeters^mm&millimeters
OBX|8|NM|SkullFetus.VP_HeadCircumference_Percentile|Fetus2|45^45%|%&percent^fmt&formatted
OBX|9|ST|BrainFetus.LateralVentricleLAppearance^Left lateral ventricle appearance|Fetus2|abnormal
OBX|10|ST|BrainFetus.LateralVentricleLDetails^Left lateral ventricle details|Fetus2|ventriculomegaly
"""
twin_vp_pps = build_viewpoint_phenopacket(twin_hl7, hpo_parser, now_ts, accession_id="TWIN")
for pp in twin_vp_pps:
    hpo_ids = {pf.type.id for pf in pp.phenotypic_features}
    print(f"{pp.id}: {hpo_ids}")

DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for head_circumference
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for biparietal_diameter
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for femur_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for abdominal_circumference
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for occipitofrontal_diameter
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for crown_rump_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for nuchal_translucency
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for tibia_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for fibula_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for radius_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for ulna_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for foot_length
DEBUG:prenatalppkt.etl.term_bin_factory:Loaded mappings for: ['head_circumference', 'biparietal_diameter', 'femur_length', 'abdominal_circumference', 'occipitofrontal_diame

twin-fetus-1: {'HP:0001321', 'HP:0000240'}
twin-fetus-2: {'HP:0002119', 'HP:0000240'}


Fetus 1's cerebellar hypoplasia (`HP:0001321`) and fetus 2's ventriculomegaly (`HP:0002119`) each stay with their own fetus. Confirmed fixed, same pattern as the Observer fix.

### A fenominal gap that needed its own synthetic message to prove

The real ViewPoint anatomy fixture that has "mild aortic arch narrowing" has no biometry data at all, so it produces zero Phenopackets through the full builder - there was no way to prove the gap survives to a real final Phenopacket without building one by hand.

In [25]:
aortic_hl7 = r"""
MSH|^~\&|ViewPoint|Hospital|||20260115130000||ORU^R01|123456|P|2.4
OBX|1|ST|Fetus.Identifier^Fetus Identifier|Fetus1|A
OBX|2|NM|SkullFetus.HeadCircumference^HC|Fetus1|175^175.0|mm&millimeters^mm&millimeters
OBX|3|NM|SkullFetus.VP_HeadCircumference_Percentile|Fetus1|50^50%|%&percent^fmt&formatted
OBX|4|ST|ChestFetus.ChestAppearance^Chest appearance|Fetus1|abnormal
OBX|5|ST|ChestFetus.ThoracicDescAortaDetails^Thoracic descending aorta details|Fetus1|mild aortic arch narrowing
"""
aortic_pps = build_viewpoint_phenopacket(aortic_hl7, hpo_parser, now_ts, accession_id="AORTIC")
hpo_ids = {pf.type.id for pf in aortic_pps[0].phenotypic_features}
print("Features present:", hpo_ids)
print("HP:0001680 (Coarctation of aorta) present:", "HP:0001680" in hpo_ids)

DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for head_circumference
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for biparietal_diameter
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for femur_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for abdominal_circumference
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for occipitofrontal_diameter
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for crown_rump_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for nuchal_translucency
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for tibia_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for fibula_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for radius_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for ulna_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for foot_length
DEBUG:prenatalppkt.etl.term_bin_factory:Loaded mappings for: ['head_circumference', 'biparietal_diameter', 'femur_length', 'abdominal_circumference', 'occipitofrontal_diame

Features present: {'HP:0000240'}
HP:0001680 (Coarctation of aorta) present: False


Confirmed absent - the real ThoracicDescAortaDetails text says "mild aortic arch narrowing," fenominal correctly matches the exact label "coarctation of the aorta" on its own, but "narrowing" alone isn't a recognized synonym and this text never says "coarctation." Same shape as Apple Sally's Dandy-Walker gap: a real, present finding, silently missing from the final record.

## 6. gyn path

A structurally different case: no fetus at all, the subject is the patient themself. `builders/gyn_phenopacket.py` reuses the same clinical-impression parsing, nothing else (gyn structured findings like adnexa/cervix/uterus have no HPO mapping yet). Tested in `test_gyn_phenopacket.py`.

In [26]:
gwen_raw = json.loads((DATA_DIR / "Gwen_Sally_pretty.json").read_text())
print("Gwen has fetuses:", gwen_raw["fetuses"])

from prenatalppkt.builders import build_gyn_phenopacket

gyn_pp = build_gyn_phenopacket(gwen_raw, hpo_parser, now_ts, accession_id="gwensally")
print("id:", gyn_pp.id, " subject:", gyn_pp.subject.id)
for pf in gyn_pp.phenotypic_features:
    print(" ", pf.type.id, pf.type.label, "excluded="+str(pf.excluded))

Gwen has fetuses: []
id: gwensally  subject: gwensally
  HP:0000126 Hydronephrosis excluded=True


`HP:0000126` (Hydronephrosis), correctly excluded - Apple Sally's real note says "No evidence of hydronephrosis," and the negation is read correctly. (A real negation bug here - `excluded` wasn't being read at all - was found and fixed earlier this session, before this rewrite.)

## 7. Genomics scaffold

Structural only - no VRS normalization, no ACMG calls, no real linkage between variants and phenotype. `genomics/vcf.py` + `genomics/genomic.py`, tested in `test_vcf.py`, `test_genomic.py`.

In [27]:
from prenatalppkt.genomics import scan_vcf_file, build_vcf_file_entry, build_genomic_interpretation

variants = scan_vcf_file(DATA_DIR / "Apple_Sally.vcf")
print(f"{len(variants)} variants, genome assembly: {variants[0].genome_assembly}")

apple_pp.files.append(
    build_vcf_file_entry(
        (DATA_DIR / "Apple_Sally.vcf").resolve().as_uri(),
        attributes={"genomeAssembly": variants[0].genome_assembly},
    )
)
apple_pp.interpretations.append(
    build_genomic_interpretation(variants, subject_id=apple_pp.subject.id, interpretation_id=f"{apple_pp.id}-interp-1")
)
print("Apple Sally's Phenopacket now has", len(apple_pp.files), "file(s) and", len(apple_pp.interpretations), "interpretation(s)")

3 variants, genome assembly: GRCh38
Apple Sally's Phenopacket now has 1 file(s) and 1 interpretation(s)


## 8. Reference growth curves

`biometry_reference.py`'s `FetalGrowthPercentiles` does real percentile computation from validated INTERGROWTH-21st and NICHD reference tables (parsed from the actual medical PDFs). Tested in `test_parse_intergrowth_txt_all.py`, `test_parse_nichd_raw.py`.

**Honest note:** the live pipeline never calls this. `etl/extractors/observer.py` only reads whatever percentile Observer/ViewPoint already pre-computed - it never independently verifies it. This module is real and tested, just not wired into the pipeline that actually runs today.

In [28]:
from prenatalppkt.biometry_reference import FetalGrowthPercentiles
from prenatalppkt.biometry_type import BiometryType

ref = FetalGrowthPercentiles(source="intergrowth")

# Note: the reference tables only have whole-week rows, so we round
# Apple Sally's 26.9-week EGA to 27 for this lookup.
independent_pct = ref.lookup_percentile(BiometryType.HEAD_CIRCUMFERENCE, 27.0, 250.0)
print(f"Independent INTERGROWTH lookup for Apple Sally's HC (250mm at 27wk): {independent_pct:.1f}%")
print(f"Observer's own pre-computed percentile: {hc['calculated_percentile']}%")
print("These are two separate computations and can disagree - the live pipeline only uses the second one.")

DEBUG:prenatalppkt.biometry_reference:Loaded measures for intergrowth: ['head_circumference', 'biparietal_diameter', 'abdominal_circumference', 'femur_length', 'occipitofrontal_diameter']


Independent INTERGROWTH lookup for Apple Sally's HC (250mm at 27wk): 47.5%
Observer's own pre-computed percentile: 42.5%
These are two separate computations and can disagree - the live pipeline only uses the second one.


## 9. Legacy Pipeline #1 - `phenotypic_export.py`

An older, disconnected pipeline (`PhenotypicExporter`, `TermObservation`, `MeasurementEvaluation`, `biometry.py`, `biometry_type.py`). Kept by deliberate decision, not fixed or hidden - here's exactly what it is and how antiquated it really is.

- **Predates the current architecture by ~7 months.** Its core shape dates to Oct 28, 2025; the real pipeline (`etl/extractors/observer.py`) didn't exist until the Nov 2025 "ETL overhaul" rewrite that superseded it.
- **Its one real runtime path has been broken the entire time.** `PhenotypicExporter.evaluate_to_observation()` calls `term_bin.category`, which crashes - `PercentileRange` was refactored at some point and this property was never updated.
- **Work still happened on it as recently as June 8, 2026** - a commit added LOINC measurement emission. But even that feature's own tests call `to_json()` with hand-built objects, going around the one broken call path - so it kept being extended without anyone exercising the path that would actually run.
- **One real, non-obsolete capability lives inside it anyway:** it's the only place `FetalGrowthPercentiles` (section 8) gets used at all.

In [29]:
from prenatalppkt.phenotypic_export import PhenotypicExporter
from prenatalppkt.biometry_type import BiometryType

exporter = PhenotypicExporter(source="intergrowth")
try:
    exporter.evaluate_to_observation(BiometryType.HEAD_CIRCUMFERENCE, 250.0, 27.0)
except AttributeError as e:
    print(f"Confirmed broken, live, using Apple Sally's own real HC value: {e}")

DEBUG:prenatalppkt.biometry_reference:Loaded measures for intergrowth: ['head_circumference', 'biparietal_diameter', 'abdominal_circumference', 'femur_length', 'occipitofrontal_diameter']
INFO:prenatalppkt.measurement_eval:Loading HPO mappings from /Users/jv2684/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/prenatalppkt/data/mappings/biometry_hpo_mappings.yaml
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for head_circumference
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for biparietal_diameter
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for femur_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for abdominal_circumference
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for occipitofrontal_diameter
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for crown_rump_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for nuchal_translucency
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for tibia_length
DEBUG:prenatalppkt.mapping_lo

Confirmed broken, live, using Apple Sally's own real HC value: 'PercentileRange' object has no attribute 'max_percentile'


## 10. Known gaps, all in one place

| Area | Gap | Status |
|---|---|---|
| Builder measurements | `build_observer_phenopacket`/`build_viewpoint_phenopacket` never emit LOINC `Measurement`s - only HPO features (section 4, 5) | Open |
| T1/Diva | Same measurements gap as above - most visible on Diva since CRL is Apply Sally's only measurement (section 4) | Open |
| Fenominal | Apple Sally's Dandy-Walker malformation - true miss in every phrasing tried, not a sentence-context issue (section 3, 4) | Open, documented as `xfail` |
| Fenominal | ViewPoint's "mild aortic arch narrowing" - recognized as an isolated exact label, not in the real sentence (section 5) | Open, documented as `xfail` |
| Fenominal | Negation inconsistency - "no evidence of agenesis of the corpus callosum" comes back `excluded=False` from fenominal itself (section 4) | Open, not yet tracked as its own test |
| LOINC codes | CRL and NT now verified; OFD still has none | 2 of 3 fixed this session |
| New measurement labels | 11 previously-unrecognized Observer labels now recognized; 5 have real percentile-bin mappings, 6 have no percentile source from Observer at all | Fixed this session (recognition); partially open (mapping) |
| Twin/multi-fetus anatomy | Both Observer and ViewPoint used to copy fetus 1's anatomy onto every fetus | Fixed this session, both formats |
| Partial/targeted scans | Used to require all four core measurements before parsing anything | Fixed this session |
| Legacy Pipeline #1 | `PhenotypicExporter`'s one real runtime path has been broken since before the Nov 2025 rewrite | Known, kept as-is by decision |
| Dead code (dto/parser, HpTerm) | Two orphaned generations of code, confirmed unused, nothing to salvage | Removed this session |